# 01 — Data and stationarity

Download SPY (2016–2024 hard cap), compute returns/target, ADF check.


In [1]:
from spy_alpha_research.data import (
    load_config, project_root, ensure_spy, save_frame, load_frame,
    price_series, compute_returns, make_target, split_by_dates,
)
ROOT = project_root()
exp = load_config(ROOT / "configs" / "experiment.yaml")
models = load_config(ROOT / "configs" / "models.yaml")
bt = load_config(ROOT / "configs" / "backtest.yaml")
FIG = ROOT / exp["paths"]["figures_dir"]
FIG.mkdir(parents=True, exist_ok=True)
PROC = ROOT / exp["paths"]["processed_dir"]


In [2]:
ohlcv = ensure_spy(
    exp["dates"]["start"], exp["dates"]["end"],
    cache_path=ROOT / exp["paths"]["raw_dir"] / "spy_ohlcv.parquet",
    auto_adjust=exp["download"]["auto_adjust"],
    progress=exp["download"]["progress"],
)
import pandas as pd
assert ohlcv.index.max() <= pd.Timestamp(exp["dates"]["end"])
px = price_series(ohlcv, exp["target"]["price_col"])
ret = compute_returns(px, exp["target"]["method"])
target = make_target(ret, exp["target"]["horizon"])
save_frame(ohlcv, PROC / "spy_ohlcv.parquet")
save_frame(ret.to_frame(), PROC / "spy_returns.parquet")
save_frame(target.to_frame(), PROC / "spy_target.parquet")
train_idx, test_idx = split_by_dates(
    ret.index, exp["dates"]["train_end"], exp["dates"]["test_start"], exp["dates"]["end"]
)
len(train_idx), len(test_idx), ret.index.min(), ret.index.max()


(1511, 752, Timestamp('2016-01-04 00:00:00'), Timestamp('2024-12-30 00:00:00'))

In [3]:
from spy_alpha_research.classical import adf_pvalue
adf_pvalue(ret.dropna())


2.5437163792692853e-27